# From Dirty Data to Model-Ready Data: Building a Preprocessing Pipeline
**Tugas Mandiri Pertemuan 2 - Data Quality Audit & Preprocessing Pipeline** (RPS 2026-CSF517 Data Mining)

**Dataset:** Titanic (klasifikasi biner, target `Survived`) - campuran fitur numerik + kategorikal + missing values + kelas tidak seimbang.

**Alur kerja (sesuai tujuan tugas):**

`Raw Data -> Data Audit -> Missing Value Handling -> Encoding -> Scaling -> Imbalance Handling -> Clean Pipeline -> Model-Ready Data`

| Bagian | Isi | Tahapan tugas |
|---|---|---|
| 1 | Setup & loading data | - |
| 2 | Data inspection, missing value audit, duplicate audit | Data Inspection, Missing Value Audit, Duplicate Audit |
| 3 | Numerical & categorical analysis | Numerical Analysis, Categorical Analysis |
| 4 | Imbalance analysis | Imbalance Analysis |
| 5 | Train-test split, ColumnTransformer pipeline | Missing Value Handling, Encoding, Scaling, Pipeline |
| 6 | Imbalance handling (class_weight / SMOTE) | Imbalance Handling |
| 7 | Eksperimen kecil A-D (KNN & Logistic Regression) | Eksperimen Kecil |
| 8 | Demonstrasi data leakage | Validation |
| 9 | Validation note & export dataset | Validation |
| 10 | Jawaban pertanyaan analisis wajib | Pertanyaan Analisis |
| 11 | Generate Executive Data Quality Report (PDF, maks. 3 halaman) | Deliverable |

> Fokus tugas adalah **kualitas dan logika preprocessing**, bukan mengejar accuracy tertinggi. Model KNN dan Logistic Regression hanya dipakai sebagai alat observasi dampak preprocessing.

## 1. Setup & Loading Data

In [ ]:

NIM = "20240801153"
NAMA = "hafiz Putra Dewanto"
RANDOM_STATE = 42

In [ ]:
import sys, os, subprocess, warnings
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Install paket otomatis jika belum ada."""
    try:
        __import__(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure("imbalanced-learn", "imblearn")
ensure("reportlab")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
os.makedirs("output", exist_ok=True)
print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
def load_titanic():
    """Cari file lokal dulu (Kaggle train.csv / titanic.csv), jika tidak ada ambil dari URL publik."""
    for p in ["titanic.csv", "train.csv", "Titanic-Dataset.csv"]:
        if os.path.exists(p):
            return pd.read_csv(p), p
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    try:
        return pd.read_csv(url), url
    except Exception as e:
        raise RuntimeError(
            "Dataset tidak dapat diunduh. Unduh titanic.csv (Kaggle: Titanic - Machine Learning from Disaster, "
            "file train.csv) lalu letakkan di folder yang sama dengan notebook ini.") from e

df_raw, SOURCE = load_titanic()
TARGET = "Survived"
print("Sumber data :", SOURCE)
print("Ukuran data :", df_raw.shape)
df_raw.head()

## 2. Data Audit
### 2.1 Data Inspection -> *Dataset profile*
Memeriksa ukuran data, tipe atribut, dan target.

In [ ]:
print(f"Jumlah baris : {df_raw.shape[0]}")
print(f"Jumlah kolom : {df_raw.shape[1]}")
print(f"Target       : {TARGET} (nilai unik: {sorted(df_raw[TARGET].unique())})")
print(f"Memori       : {df_raw.memory_usage(deep=True).sum()/1024:.1f} KB\n")

profile = pd.DataFrame({
    "Dtype": df_raw.dtypes.astype(str),
    "Non-null": df_raw.notna().sum(),
    "Missing": df_raw.isna().sum(),
    "Missing %": (df_raw.isna().mean() * 100).round(2),
    "Unique": df_raw.nunique(),
    "Contoh nilai": [df_raw[c].dropna().iloc[0] for c in df_raw.columns],
})
display(profile)

**Komentar.** Dataset berisi kolom numerik (`Age`, `SibSp`, `Parch`, `Fare`), kategorikal (`Pclass`, `Sex`, `Embarked`), serta kolom identitas/teks berkardinalitas tinggi (`PassengerId`, `Name`, `Ticket`, `Cabin`). Perhatikan bahwa `Pclass` bertipe integer tetapi **secara semantik kategorikal ordinal**; tipe data saja tidak cukup untuk menentukan jenis fitur.

### 2.2 Missing Value Audit -> *Missing value table*

In [ ]:
missing_table = (pd.DataFrame({"Jumlah missing": df_raw.isna().sum(),
                               "Persentase (%)": (df_raw.isna().mean() * 100).round(2)})
                 .query("`Jumlah missing` > 0")
                 .sort_values("Jumlah missing", ascending=False))
display(missing_table)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.barplot(x=missing_table["Persentase (%)"], y=missing_table.index, ax=ax[0], color="#C0504D")
ax[0].set_title("Persentase missing value per atribut")
for i, v in enumerate(missing_table["Persentase (%)"]):
    ax[0].text(v + 0.8, i, f"{v:.1f}%", va="center")
sns.heatmap(df_raw.isna(), cbar=False, yticklabels=False, ax=ax[1], cmap="Reds")
ax[1].set_title("Peta missing value (merah = missing)")
plt.tight_layout(); plt.show()

**Komentar.**
- `Cabin` memiliki missing paling besar (> 75%): imputasi tidak masuk akal karena akan mengarang mayoritas nilai.
- `Age` memiliki missing sekitar 20%: cukup besar sehingga perlu diimputasi, bukan baris dibuang (membuang ~20% data akan mengurangi sampel dan dapat bias).
- `Embarked` hanya 2 baris missing: dampak sangat kecil, imputasi modus sudah memadai.
- Pola missing pada `Cabin` sendiri informatif (kabin umumnya tercatat untuk penumpang kelas 1) sehingga dibuat fitur biner `HasCabin`.

### 2.3 Duplicate Audit -> *Jumlah duplikat*

In [ ]:
dup_full = df_raw.duplicated().sum()
dup_noid = df_raw.drop(columns="PassengerId").duplicated().sum()
dup_name = df_raw["Name"].duplicated().sum()
dup_ticket = df_raw["Ticket"].duplicated().sum()

dup_table = pd.DataFrame({
    "Pemeriksaan": ["Duplikat baris penuh", "Duplikat baris (tanpa PassengerId)", "Nama duplikat", "Ticket duplikat"],
    "Jumlah": [dup_full, dup_noid, dup_name, dup_ticket],
    "Interpretasi": ["Tidak ada record ganda" if dup_full == 0 else "Perlu dihapus",
                     "Tidak ada record ganda tersembunyi" if dup_noid == 0 else "Periksa manual",
                     "Setiap penumpang unik" if dup_name == 0 else "Periksa manual",
                     "Bukan duplikat data: penumpang satu keluarga/grup berbagi tiket"]})
display(dup_table)
df_raw = df_raw.drop_duplicates().reset_index(drop=True)
print("Ukuran data setelah drop_duplicates:", df_raw.shape)

**Komentar.** Duplikat `Ticket` bukan duplikasi record, melainkan penumpang yang bepergian bersama. Hal ini penting untuk bagian **data leakage**: anggota keluarga bisa terpisah di train dan test (group leakage).

## 3. Numerical & Categorical Analysis

### 3.1 Numerical Analysis -> *Summary statistics*

In [ ]:
NUM_RAW = ["Age", "SibSp", "Parch", "Fare"]
summary = df_raw[NUM_RAW].describe().T
summary["skew"] = df_raw[NUM_RAW].skew()
summary["kurtosis"] = df_raw[NUM_RAW].kurt()
display(summary.round(3))

def iqr_outliers(s):
    s = s.dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum())
out_table = pd.DataFrame({"Outlier (aturan IQR)": {c: iqr_outliers(df_raw[c]) for c in NUM_RAW}})
out_table["% dari data"] = (out_table["Outlier (aturan IQR)"] / len(df_raw) * 100).round(2)
display(out_table)

fig, ax = plt.subplots(2, 4, figsize=(14, 5.5))
for i, c in enumerate(NUM_RAW):
    sns.histplot(df_raw[c].dropna(), kde=True, ax=ax[0, i], color="#4F81BD"); ax[0, i].set_title(f"Distribusi {c}")
    sns.boxplot(x=df_raw[c], ax=ax[1, i], color="#9BBB59"); ax[1, i].set_title(f"Boxplot {c}")
plt.tight_layout(); plt.show()

**Komentar.**
- Rentang antar fitur sangat berbeda: `Fare` 0 s.d. ~512, `Age` 0 s.d. 80, `SibSp`/`Parch` kecil -> **perlu scaling** untuk algoritma berbasis jarak/gradien.
- `Fare` sangat *right-skewed* (skewness besar) dengan banyak outlier tinggi. Outlier **tidak dihapus**: tarif mahal adalah penumpang kelas 1 yang valid secara bisnis, bukan error. Scaling dengan StandardScaler tetap mengubah skala, tetapi bentuk distribusi tidak berubah.
- Perbedaan mean vs median `Age` kecil, sehingga median aman dipakai sebagai nilai imputasi.

### 3.2 Categorical Analysis -> *Category summary*

In [ ]:
CAT_RAW = ["Pclass", "Sex", "Embarked", "Cabin", "Ticket", "Name"]
cat_type = {"Pclass": "Ordinal (1 < 2 < 3)", "Sex": "Nominal", "Embarked": "Nominal",
            "Cabin": "Nominal, kardinalitas tinggi", "Ticket": "Identitas / kardinalitas tinggi",
            "Name": "Identitas / teks"}
cat_summary = pd.DataFrame({
    "Jenis": pd.Series(cat_type),
    "Unique": {c: df_raw[c].nunique() for c in CAT_RAW},
    "Missing": {c: int(df_raw[c].isna().sum()) for c in CAT_RAW},
    "Modus": {c: df_raw[c].mode().iloc[0] for c in CAT_RAW},
    "Frekuensi modus": {c: int(df_raw[c].value_counts().iloc[0]) for c in CAT_RAW},
}).loc[CAT_RAW]
display(cat_summary)

for c in ["Pclass", "Sex", "Embarked"]:
    t = df_raw.groupby(c)[TARGET].agg(Jumlah="count", Survival_rate="mean").round(3)
    print(f"\nFrekuensi & survival rate per {c}:"); display(t)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, c in zip(ax, ["Pclass", "Sex", "Embarked"]):
    sns.barplot(data=df_raw, x=c, y=TARGET, ax=a, color="#4BACC6", errorbar=None)
    a.set_title(f"Survival rate per {c}"); a.set_ylabel("Rata-rata Survived")
plt.tight_layout(); plt.show()

**Komentar.**
- **Nominal:** `Sex`, `Embarked` (tidak ada urutan) -> One-Hot Encoding. (`Cabin` juga nominal tetapi kardinalitas tinggi dan missing > 75%.)
- **Ordinal:** `Pclass` (kelas 1 > 2 > 3) -> Ordinal Encoding agar urutan dipertahankan dengan hanya satu kolom.
- **Identitas:** `PassengerId`, `Name`, `Ticket` bernilai (hampir) unik; jika di-encode akan menyebabkan overfitting dan tidak memiliki daya generalisasi, sehingga dibuang.
- Grafik survival rate menunjukkan `Sex` dan `Pclass` sangat berpengaruh terhadap target, sehingga fitur ini wajib dipertahankan dan di-encode dengan benar.

## 4. Imbalance Analysis -> *Class distribution*

In [ ]:
class_counts = df_raw[TARGET].value_counts().sort_index()
class_pct = (df_raw[TARGET].value_counts(normalize=True).sort_index() * 100).round(2)
class_dist = pd.DataFrame({"Jumlah": class_counts, "Persentase (%)": class_pct})
class_dist.index = ["0 = Meninggal", "1 = Selamat"]
display(class_dist)

ratio = class_counts.max() / class_counts.min()
print(f"Rasio mayoritas : minoritas = {ratio:.2f} : 1")
print(f"Proporsi kelas minoritas   = {class_counts.min()/len(df_raw)*100:.1f}%")

plt.figure(figsize=(4.5, 3.2))
sns.barplot(x=class_dist.index, y=class_dist["Jumlah"], palette=["#4F81BD", "#9BBB59"])
for i, v in enumerate(class_dist["Jumlah"]): plt.text(i, v + 5, str(v), ha="center")
plt.title("Distribusi kelas target"); plt.tight_layout(); plt.show()

**Komentar.** Rasio kelas sekitar **1.6 : 1** (kelas minoritas ~38%). Ini termasuk **imbalance ringan**: belum ekstrem (seperti fraud 1:100), tetapi cukup untuk menggeser prediksi model ke kelas mayoritas. Karena itu tetap digunakan **stratified split** (menjaga proporsi) dan imbalance handling dibandingkan secara eksperimen (bagian 7). Accuracy saja tidak cukup, sehingga Precision, Recall, F1, dan ROC-AUC dilaporkan.

## 5. Preprocessing Pipeline
### 5.1 Pembersihan awal yang aman (tanpa leakage)
Dua transformasi berikut bersifat **per-baris** (tidak memakai target maupun statistik dari baris lain), sehingga aman dilakukan sebelum split:
- `FamilySize = SibSp + Parch + 1`
- `HasCabin` = 1 jika `Cabin` terisi (menggantikan `Cabin` yang missing > 75%)
- Drop `PassengerId`, `Name`, `Ticket`, `Cabin` (identitas / kardinalitas tinggi).

Semua langkah yang **mempelajari statistik data** (median, modus, mean, std, kategori, SMOTE) **sengaja ditunda** sampai setelah train-test split.

In [ ]:
def basic_cleaning(d):
    d = d.copy()
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
    d["HasCabin"] = d["Cabin"].notna().astype(int)
    return d.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

df_fe = basic_cleaning(df_raw)
print("Kolom setelah pembersihan awal:", list(df_fe.columns))
display(df_fe.head())

### 5.2 Train-Test Split (dilakukan **sebelum** imputasi, scaling, encoding, dan SMOTE)
Gunakan `stratify=y` agar proporsi kelas pada train dan test sama.

In [ ]:
X = df_fe.drop(columns=TARGET)
y = df_fe[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

split_info = pd.DataFrame({
    "Jumlah baris": [len(X), len(X_train), len(X_test)],
    "Kelas 0": [(y == 0).sum(), (y_train == 0).sum(), (y_test == 0).sum()],
    "Kelas 1": [(y == 1).sum(), (y_train == 1).sum(), (y_test == 1).sum()],
    "% Kelas 1": [y.mean() * 100, y_train.mean() * 100, y_test.mean() * 100]},
    index=["Total", "Train", "Test"]).round(2)
display(split_info)

### 5.3 Rancangan ColumnTransformer
| Kelompok | Kolom | Imputasi | Transformasi | Alasan |
|---|---|---|---|---|
| Numerik | Age, Fare, SibSp, Parch, FamilySize | median | StandardScaler | median robust terhadap outlier/skew; skala berbeda |
| Ordinal | Pclass | most_frequent | OrdinalEncoder (1<2<3) | urutan kelas bermakna |
| Nominal | Sex, Embarked | most_frequent | OneHotEncoder | tidak ada urutan; `handle_unknown="ignore"` agar aman pada data baru |
| Biner | HasCabin | - | passthrough | sudah 0/1 |

Struktur sesuai ketentuan tugas: *Raw -> Train-Test Split -> (Numerical: Imputation+Scaling) + (Categorical: Imputation+Encoding) -> ColumnTransformer -> Imbalance Handling on Training Data -> Model-Ready Dataset*.

In [ ]:
NUM_COLS = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
ORD_COLS = ["Pclass"]
NOM_COLS = ["Sex", "Embarked"]
BIN_COLS = ["HasCabin"]

def build_preprocessor(scaler="standard"):
    """scaler: 'standard' | 'minmax' | 'none'  (untuk eksperimen A-D)."""
    sc = {"standard": StandardScaler(), "minmax": MinMaxScaler(), "none": "passthrough"}[scaler]
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", sc)])
    ord_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("encoder", OrdinalEncoder(categories=[[1, 2, 3]]))])
    nom_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("encoder", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False))])
    prep = ColumnTransformer([("num", num_pipe, NUM_COLS),
                              ("ord", ord_pipe, ORD_COLS),
                              ("nom", nom_pipe, NOM_COLS),
                              ("bin", "passthrough", BIN_COLS)],
                             remainder="drop", verbose_feature_names_out=False)
    prep.set_output(transform="pandas")
    return prep

preprocessor = build_preprocessor("standard")
preprocessor   # tampilan diagram pipeline di Jupyter

In [ ]:
# fit HANYA pada train, lalu transform train dan test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Shape train:", X_train_prep.shape, "| Shape test:", X_test_prep.shape)
print("Kolom hasil:", list(X_train_prep.columns))
print("Total NaN train/test:", int(X_train_prep.isna().sum().sum()), "/", int(X_test_prep.isna().sum().sum()))
display(X_train_prep.head())
display(X_train_prep.describe().T.round(3))

In [ ]:
# Dampak imputasi: parameter yang dipelajari dari TRAIN
age_median_train = X_train["Age"].median()
print(f"Median Age (train) : {age_median_train:.2f} | Mean Age (train): {X_train['Age'].mean():.2f}")
print("Modus Embarked (train):", X_train["Embarked"].mode()[0])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
sns.histplot(X_train["Age"].dropna(), bins=30, ax=ax[0], color="#4F81BD"); ax[0].set_title("Age sebelum imputasi")
sns.histplot(X_train["Age"].fillna(age_median_train), bins=30, ax=ax[1], color="#F79646"); ax[1].set_title("Age sesudah imputasi median")
plt.tight_layout(); plt.show()

**Komentar imputasi.** Setelah imputasi median muncul lonjakan di sekitar nilai median (efek yang wajar dan diketahui dari imputasi konstanta). Sebaran umum Age tidak berubah drastis. Alternatif yang lebih canggih (median per `Pclass`/`Sex`, atau `KNNImputer`) dapat dicoba, namun median global dipilih karena sederhana, stabil, dan mudah diaudit.

## 6. Imbalance Handling -> *Balanced training data*
Dua strategi dibandingkan: **`class_weight='balanced'`** (menaikkan bobot kesalahan kelas minoritas, tanpa mengubah data) dan **SMOTE** (membuat sampel sintetis minoritas dengan interpolasi tetangga terdekat). Keduanya **hanya diterapkan pada data train** setelah preprocessing.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_prep, y_train)

before = y_train.value_counts().sort_index()
after = pd.Series(y_train_bal).value_counts().sort_index()
bal_table = pd.DataFrame({"Train asli": before, "Train + SMOTE": after, "Test (tidak diubah)": y_test.value_counts().sort_index()})
bal_table.index = ["0 = Meninggal", "1 = Selamat"]
display(bal_table)
print("Shape train setelah SMOTE:", X_train_bal.shape)

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for a, s, t in zip(ax, [before, after], ["Sebelum SMOTE (train)", "Sesudah SMOTE (train)"]):
    sns.barplot(x=["Meninggal", "Selamat"], y=s.values, ax=a, palette=["#4F81BD", "#9BBB59"]); a.set_title(t)
    for i, v in enumerate(s.values): a.text(i, v + 3, str(v), ha="center")
plt.tight_layout(); plt.show()

**Komentar.** SMOTE hanya menambah sampel kelas minoritas **di train**; test tetap utuh agar evaluasi mencerminkan distribusi dunia nyata. Catatan teknis: SMOTE standar bekerja pada ruang numerik kontinu sehingga kolom one-hot hasil interpolasi bisa bernilai pecahan (mis. 0.37). Untuk data dengan banyak fitur kategorikal, `SMOTENC` adalah alternatif yang lebih tepat; di sini efeknya kecil karena hanya 2 fitur nominal.

## 7. Eksperimen Kecil: Dampak Preprocessing
Perbandingan kondisi sesuai tugas:

| Eksperimen | Preprocessing | Catatan |
|---|---|---|
| A | Tanpa scaling | Baseline |
| B | Dengan StandardScaler (+ MinMaxScaler sebagai pembanding, B2) | Analisis perubahan |
| C | Tanpa imbalance handling | Baseline distribusi kelas |
| D | class_weight (D1) / SMOTE (D2) | Analisis perubahan distribusi |

Eksperimen C memakai konfigurasi preprocessing terbaik dari analisis scaling (StandardScaler) tanpa penanganan imbalance, sehingga identik secara konfigurasi dengan B. Ia dihitung ulang sebagai baseline pasangan C vs D.
KNN tidak memiliki parameter `class_weight`, sehingga D1 hanya untuk Logistic Regression. Selain hold-out test, dilaporkan juga **5-fold stratified CV pada data train**; di setiap fold, imputer/scaler/SMOTE di-*fit* hanya pada fold-train (tanpa leakage).

In [ ]:
def make_model(name, class_weight=None):
    if name == "LogReg":
        return LogisticRegression(max_iter=1000, class_weight=class_weight, random_state=RANDOM_STATE)
    return KNeighborsClassifier(n_neighbors=15)

def run_experiment(model_name, scaler, imb, X_tr, y_tr, X_te, y_te):
    """Fit preprocessing pada X_tr SAJA, (opsional) imbalance handling pada data train, evaluasi pada X_te."""
    prep = build_preprocessor(scaler)
    Xt = prep.fit_transform(X_tr)
    Xe = prep.transform(X_te)
    yt, cw = y_tr, None
    if imb == "class_weight":
        cw = "balanced"
    elif imb == "smote":
        Xt, yt = SMOTE(random_state=RANDOM_STATE).fit_resample(Xt, y_tr)
    model = make_model(model_name, cw).fit(Xt, yt)
    pred, proba = model.predict(Xe), model.predict_proba(Xe)[:, 1]
    return {"Accuracy": accuracy_score(y_te, pred),
            "Precision": precision_score(y_te, pred, zero_division=0),
            "Recall": recall_score(y_te, pred, zero_division=0),
            "F1": f1_score(y_te, pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_te, proba),
            "Pred_Pos_Rate": float(np.mean(pred)),
            "Train_Pos_Rate": float(np.mean(yt))}

def cv_f1(model_name, scaler, imb, X, y, k=5):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    s = [run_experiment(model_name, scaler, imb, X.iloc[a], y.iloc[a], X.iloc[b], y.iloc[b])["F1"]
         for a, b in skf.split(X, y)]
    return float(np.mean(s)), float(np.std(s))

EXPS = [("A",  "Tanpa scaling (baseline)",            "none",     "none"),
        ("B",  "StandardScaler",                      "standard", "none"),
        ("B2", "MinMaxScaler",                        "minmax",   "none"),
        ("C",  "Tanpa imbalance handling (baseline)", "standard", "none"),
        ("D1", "class_weight='balanced'",             "standard", "class_weight"),
        ("D2", "SMOTE (train only)",                  "standard", "smote")]

rows = []
for exp, desc, sc, imb in EXPS:
    for m in ["LogReg", "KNN"]:
        if imb == "class_weight" and m == "KNN":
            continue                       # KNN tidak punya class_weight
        r = run_experiment(m, sc, imb, X_train, y_train, X_test, y_test)
        mu, sd = cv_f1(m, sc, imb, X_train, y_train)
        rows.append({"Exp": exp, "Model": m, "Preprocessing": desc, **r, "CV_F1_mean": mu, "CV_F1_std": sd})
res = pd.DataFrame(rows)
display(res.round(3))
res.round(4).to_csv("output/hasil_eksperimen.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, col in zip(ax, ["F1", "Recall", "Precision"]):
    pv = res.pivot(index="Exp", columns="Model", values=col)
    pv.plot(kind="bar", ax=a, rot=0, color=["#F79646", "#4BACC6"]); a.set_ylim(0, 1); a.set_title(f"{col} (test set)")
plt.tight_layout(); plt.show()

# analisis perubahan distribusi prediksi
dist = res[res.Exp.isin(["C", "D1", "D2"])][["Exp", "Model", "Train_Pos_Rate", "Pred_Pos_Rate", "Recall", "Precision"]]
dist = dist.assign(Test_True_Pos_Rate=float(y_test.mean()))
print("Perubahan distribusi kelas (train) dan distribusi prediksi (test):")
display(dist.round(3))

### Interpretasi eksperimen
Tuliskan interpretasi Anda setelah melihat angka di atas, menjawab minimal:
1. **Scaling (A vs B):** model mana yang berubah paling besar? Mengapa KNN lebih sensitif dibanding Logistic Regression? (jarak Euclidean didominasi `Fare` bila tidak diskalakan)
2. **Imbalance (C vs D):** apakah Recall kelas minoritas naik? Apakah Precision/Accuracy turun? Perhatikan `Pred_Pos_Rate` yang mendekati/menjauhi proporsi kelas positif sebenarnya.
3. **Trade-off:** karena imbalance hanya ringan (~1.6:1), perubahan metrik bisa kecil; ini wajar dan sebaiknya dilaporkan apa adanya, bukan dipaksa terlihat dramatis.
4. Perbandingan nilai hold-out dan CV: apakah selisihnya kecil (stabil) atau besar (varians tinggi)?

(Laporan PDF pada bagian 11 membuat interpretasi otomatis berdasarkan angka hasil run Anda.)

## 8. Data Leakage: Identifikasi & Demonstrasi
Identifikasi kemungkinan leakage pada preprocessing:

| # | Sumber leakage | Mengapa bocor | Pencegahan di notebook ini |
|---|---|---|---|
| 1 | `fit` imputer / scaler / encoder pada **seluruh data** sebelum split | median, mean, std, dan kategori test ikut menentukan parameter train | split lebih dulu; `ColumnTransformer.fit` hanya pada train |
| 2 | **SMOTE sebelum split** | sampel sintetis (interpolasi dari sampel train) masuk ke test, sehingga model "mengenal" test | SMOTE hanya pada `X_train_prep` |
| 3 | Fitur turunan **target** / target encoding tanpa CV | informasi label masuk ke fitur | tidak ada fitur turunan `Survived` |
| 4 | **Group leakage**: keluarga/tiket sama terpisah di train dan test | model hafal pola keluarga | dicatat sebagai keterbatasan (bisa diatasi dengan `GroupShuffleSplit` berdasarkan Ticket) |
| 5 | Cross-validation dengan preprocessing di luar loop | statistik fold-validasi bocor | preprocessing dan SMOTE dikerjakan ulang di setiap fold (`cv_f1`) |

Berikut demonstrasi nyata untuk #1 dan #2.

In [ ]:
# (1) Preprocessing bocor: fit pada seluruh X
leaky_prep = build_preprocessor("standard").fit(X)
scaler_leak = leaky_prep.named_transformers_["num"]["scaler"]
scaler_ok = preprocessor.named_transformers_["num"]["scaler"]
imp_leak = leaky_prep.named_transformers_["num"]["imputer"].statistics_
imp_ok = preprocessor.named_transformers_["num"]["imputer"].statistics_

cmp1 = pd.DataFrame({"Median impute (benar: train saja)": imp_ok, "Median impute (bocor: semua data)": imp_leak,
                     "Mean scaler (benar)": scaler_ok.mean_, "Mean scaler (bocor)": scaler_leak.mean_,
                     "Std scaler (benar)": scaler_ok.scale_, "Std scaler (bocor)": scaler_leak.scale_}, index=NUM_COLS)
print("Parameter preprocessing: prosedur benar vs bocor")
display(cmp1.round(3))

# (2) SMOTE sebelum split (bocor) vs setelah split (benar), menggunakan KNN yang paling sensitif
X_all_prep = build_preprocessor("standard").fit_transform(X)
X_res, y_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_all_prep, y)
Xa, Xb, ya, yb = train_test_split(X_res, y_res, test_size=0.20, stratify=y_res, random_state=RANDOM_STATE)
knn_leak = make_model("KNN").fit(Xa, ya)
leak_f1 = f1_score(yb, knn_leak.predict(Xb))
leak_acc = accuracy_score(yb, knn_leak.predict(Xb))
clean = res[(res.Exp == "D2") & (res.Model == "KNN")].iloc[0]
clean_f1, clean_acc = float(clean["F1"]), float(clean["Accuracy"])

leak_table = pd.DataFrame({"Prosedur": ["SMOTE SESUDAH split (benar)", "SMOTE SEBELUM split (bocor)"],
                           "Accuracy": [clean_acc, leak_acc], "F1": [clean_f1, leak_f1]}).round(3)
display(leak_table)
print(f"Selisih F1 akibat leakage: {leak_f1 - clean_f1:+.3f}")

**Komentar.** Pada demonstrasi #2, skor KNN dengan SMOTE-sebelum-split terlihat lebih tinggi karena sebagian "test" adalah titik sintetis yang berada di antara/dekat titik train sehingga mudah ditebak. Skor ini **menyesatkan** dan tidak akan terulang pada data baru. Pada #1, perbedaan parameter terlihat kecil karena dataset cukup besar dan split acak; pada data kecil atau berdistribusi bergeser, perbedaan bisa lebih besar. Prinsipnya tetap: test set tidak boleh mempengaruhi apa pun yang dipelajari saat training.

## 9. Validation -> *Validation note* & Export dataset siap model

In [ ]:
checks = []
def check(name, cond, detail=""):
    checks.append({"Pemeriksaan": name, "Hasil": "LULUS" if cond else "GAGAL", "Detail": detail})

check("Tidak ada NaN pada train & test hasil preprocessing",
      X_train_prep.isna().sum().sum() == 0 and X_test_prep.isna().sum().sum() == 0)
check("Kolom train dan test identik", list(X_train_prep.columns) == list(X_test_prep.columns),
      f"{X_train_prep.shape[1]} fitur")
check("Tidak ada indeks yang beririsan antara train dan test",
      len(set(X_train.index) & set(X_test.index)) == 0)
check("Split stratified (selisih proporsi kelas < 1 poin persen)",
      abs(y_train.mean() - y_test.mean()) < 0.01, f"train {y_train.mean():.3f} vs test {y_test.mean():.3f}")
num_means = X_train_prep[NUM_COLS].mean().abs().max(); num_stds = (X_train_prep[NUM_COLS].std(ddof=0) - 1).abs().max()
check("Train ter-standardisasi (mean ~ 0, std ~ 1)", num_means < 1e-6 and num_stds < 1e-6)
check("Test TIDAK persis ter-standardisasi (bukti scaler tidak di-fit pada test)",
      X_test_prep[NUM_COLS].mean().abs().max() > 1e-6,
      f"mean test maksimum = {X_test_prep[NUM_COLS].mean().abs().max():.3f}")
check("Statistik imputasi Age = median train", np.isclose(preprocessor.named_transformers_["num"]["imputer"].statistics_[0],
      X_train["Age"].median()))
check("SMOTE hanya mengubah train (jumlah test tetap)", len(X_test_prep) == len(y_test) and len(X_train_bal) > len(X_train_prep))
check("Kelas train seimbang setelah SMOTE", pd.Series(y_train_bal).value_counts().nunique() == 1)
check("Pipeline reproducible (dua kali fit menghasilkan keluaran identik)",
      np.allclose(build_preprocessor("standard").fit_transform(X_train).values, X_train_prep.values))
validation = pd.DataFrame(checks)
display(validation)
assert (validation["Hasil"] == "LULUS").all(), "Ada pemeriksaan validasi yang gagal!"
print("\nSemua pemeriksaan validasi lulus.")

In [ ]:
# Export dataset model-ready + pipeline
import joblib
X_train_prep.assign(**{TARGET: y_train.values}).to_csv("output/train_ready_original.csv", index=False)
pd.DataFrame(X_train_bal, columns=X_train_prep.columns).assign(**{TARGET: np.asarray(y_train_bal)}).to_csv("output/train_ready_smote.csv", index=False)
X_test_prep.assign(**{TARGET: y_test.values}).to_csv("output/test_ready.csv", index=False)
joblib.dump(preprocessor, "output/preprocessor.joblib")
print("Tersimpan di folder output/:", sorted(os.listdir("output")))

### Validation Note
- Seluruh langkah yang mempelajari statistik data (imputasi, scaling, encoding, SMOTE) dijalankan **setelah** split dan hanya di-*fit* pada **train**. Test hanya melalui `transform`.
- Transformasi per-baris (`FamilySize`, `HasCabin`) tidak memakai target dan tidak memakai baris lain, sehingga tidak menimbulkan leakage meskipun dilakukan sebelum split.
- Pipeline `ColumnTransformer` bersifat reproducible (`random_state` tetap; hasil fit ulang identik) dan dapat disimpan (`preprocessor.joblib`) untuk dipakai pada data baru.
- Keterbatasan: group leakage (keluarga/tiket sama) belum ditangani; SMOTE standar pada kolom one-hot menghasilkan nilai pecahan; imputasi median global tidak mempertimbangkan subkelompok.

## 10. Jawaban Pertanyaan Analisis Wajib
> Angka merujuk pada dataset Titanic (891 baris); verifikasi dengan output di atas bila memakai versi data lain.

**1. Missing Values.** Berapa jumlah missing pada masing-masing atribut? Strategi apa dan mengapa?
- `Age`: 177 baris (~19.9%) -> **imputasi median** (dari train). Median robust terhadap outlier; membuang 20% baris akan mengurangi data dan berpotensi menimbulkan bias.
- `Cabin`: 687 baris (~77.1%) -> **dibuang**, diganti flag biner `HasCabin`. Imputasi terhadap >75% nilai hanya mengarang data; ada/tidaknya kabin tetap informatif.
- `Embarked`: 2 baris (~0.2%) -> **imputasi modus**. Dampak sangat kecil dan fitur nominal.
- Atribut lain tidak memiliki missing value.

**2. Categorical Features.** Atribut mana yang nominal dan ordinal? Teknik encoding apa?
- **Nominal:** `Sex`, `Embarked` (juga `Cabin`) -> **One-Hot Encoding** (menghindari urutan palsu).
- **Ordinal:** `Pclass` (1 > 2 > 3) -> **Ordinal Encoding** dengan urutan eksplisit.
- `Name`, `Ticket`, `PassengerId` adalah identitas/teks berkardinalitas tinggi -> dibuang.

**3. Feature Scaling.** Fitur mana yang perlu di-scaling? Mengapa penting untuk algoritma tertentu tetapi tidak selalu kritis untuk tree?
- Di-scaling: `Age`, `Fare`, `SibSp`, `Parch`, `FamilySize` (StandardScaler). Fitur ordinal/biner/one-hot sudah berskala kecil.
- **KNN, SVM, regresi dengan regularisasi/gradient descent, dan PCA** bergantung pada jarak/besar koefisien sehingga fitur berskala besar (`Fare`) mendominasi bila tidak diskalakan. **Decision Tree/Random Forest/Gradient Boosting** hanya memakai pemisahan berdasarkan ambang per fitur (urutan nilai), sehingga invarian terhadap transformasi monoton seperti scaling.

**4. Class Imbalance.** Berapa rasio kelas target? Apakah imbalanced? Strategi?
- 549 meninggal : 342 selamat = **~1.61 : 1** (minoritas ~38.4%) -> **imbalance ringan**.
- Strategi: stratified split, lalu `class_weight='balanced'` dan **SMOTE pada train**, keduanya dibandingkan dengan baseline; metrik evaluasi: Precision, Recall, F1, ROC-AUC (bukan hanya accuracy).

**5. SMOTE.** Mengapa hanya pada training data, bukan sebelum train-test split?
- SMOTE membuat titik sintetis dari interpolasi tetangga. Jika dijalankan sebelum split, titik sintetis (dan titik aslinya) tersebar ke train dan test sehingga test berisi "turunan" data train -> model dievaluasi pada data yang sudah dikenalnya (**leakage**) dan skor terlalu optimistis (bukti pada bagian 8). Test harus tetap merepresentasikan distribusi asli dunia nyata.

**6. Data Leakage.** Minimal dua kemungkinan leakage pada preprocessing:
1. Fit imputer/scaler/encoder pada seluruh data sebelum split (statistik test ikut mempengaruhi train).
2. SMOTE sebelum split.
3. (Tambahan) Fitur turunan target / target encoding tanpa CV; group leakage keluarga/tiket yang terpisah di train dan test; preprocessing di luar loop cross-validation.

## 11. Executive Data Quality Report (PDF, maksimal 3 halaman)
Sel berikut mengumpulkan seluruh angka dari notebook ini dan membuat laporan **`P02_<NIM>_<Nama>_Report.pdf`** dengan alur: *Data Profile -> Data Quality Problems -> Treatment -> Justification -> Final Dataset Condition*. Jumlah halaman diperiksa otomatis (maksimal 3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image

NAVY = "#1F3864"


def make_fig(info, res, path):
    """Satu gambar 3 panel: missing value | distribusi kelas | F1 per eksperimen."""
    fig, ax = plt.subplots(1, 3, figsize=(7.6, 2.3))
    # (1) missing value
    mc = info["miss"]
    ax[0].barh(list(mc.keys()), list(mc.values()), color="#C0504D")
    for i, v in enumerate(mc.values()):
        ax[0].text(v + 1, i, f"{v:.1f}%", va="center", fontsize=7)
    ax[0].set_xlim(0, max(mc.values()) * 1.35)
    ax[0].set_title("Missing value (%)", fontsize=8)
    ax[0].invert_yaxis()
    # (2) distribusi kelas
    lab = ["Train asli", "Train + SMOTE"]
    neg = [info["tr_neg"], info["sm_neg"]]
    pos = [info["tr_pos"], info["sm_pos"]]
    ax[1].bar(lab, neg, color="#4F81BD", label="Meninggal (0)")
    ax[1].bar(lab, pos, bottom=neg, color="#9BBB59", label="Selamat (1)")
    for i in range(2):
        ax[1].text(i, neg[i] / 2, str(neg[i]), ha="center", color="white", fontsize=7)
        ax[1].text(i, neg[i] + pos[i] / 2, str(pos[i]), ha="center", fontsize=7)
    ax[1].set_title("Distribusi kelas (train)", fontsize=8)
    ax[1].legend(fontsize=6, loc="lower right")
    # (3) F1 per eksperimen
    if res is not None:
        pv = res.pivot(index="Exp", columns="Model", values="F1")
        pv.plot(kind="bar", ax=ax[2], color=["#F79646", "#4BACC6"], width=0.75, rot=0)
        ax[2].set_ylim(0, 1)
        ax[2].legend(fontsize=6)
        ax[2].set_xlabel("")
        ax[2].set_title("F1 test per eksperimen", fontsize=8)
    else:
        ax[2].axis("off")
        ax[2].text(0.5, 0.5, "Panel F1 eksperimen\nterisi otomatis setelah\nnotebook dijalankan",
                   ha="center", va="center", fontsize=8, color="gray")
    for a in ax:
        a.tick_params(labelsize=7)
    plt.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def build_report(path, info, res, fig_path, nim="NIM", nama="Nama"):
    base = ParagraphStyle("b", fontName="Helvetica", fontSize=8.1, leading=10, alignment=4)
    sm = ParagraphStyle("sm", parent=base, fontSize=7, leading=8.4, alignment=0)
    smb = ParagraphStyle("smb", parent=sm, fontName="Helvetica-Bold", textColor=colors.white)
    h1 = ParagraphStyle("h1", fontName="Helvetica-Bold", fontSize=9.5, leading=11.5,
                        textColor=colors.HexColor(NAVY), spaceBefore=5, spaceAfter=2)
    ttl = ParagraphStyle("t", fontName="Helvetica-Bold", fontSize=13, leading=16,
                         textColor=colors.HexColor(NAVY))
    sub = ParagraphStyle("sub", fontName="Helvetica", fontSize=8, leading=10, textColor=colors.grey)
    note = ParagraphStyle("note", parent=base, backColor=colors.HexColor("#FFF2CC"),
                          borderPadding=3, fontSize=7.6)

    def P(t, s=base):
        return Paragraph(t, s)

    def tbl(data, widths, zebra=True):
        rows = [[P(str(c), smb if i == 0 else sm) for c in r] for i, r in enumerate(data)]
        t = Table(rows, colWidths=[w * cm for w in widths], repeatRows=1)
        st = [("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(NAVY)),
              ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#A6A6A6")),
              ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
              ("TOPPADDING", (0, 0), (-1, -1), 1.6), ("BOTTOMPADDING", (0, 0), (-1, -1), 1.6),
              ("LEFTPADDING", (0, 0), (-1, -1), 3), ("RIGHTPADDING", (0, 0), (-1, -1), 3)]
        if zebra:
            for i in range(2, len(rows), 2):
                st.append(("BACKGROUND", (0, i), (-1, i), colors.HexColor("#F2F2F2")))
        t.setStyle(TableStyle(st))
        return t

    def delta_word(a, b, tol=0.005):
        d = b - a
        return "relatif tetap" if abs(d) < tol else ("naik" if d > 0 else "turun")

    def g(exp, model, col):
        if res is None:
            return float("nan")
        r = res[(res.Exp == exp) & (res.Model == model)]
        return float(r[col].iloc[0]) if len(r) else float("nan")

    W = 18.2  # lebar konten (cm)
    S = []
    S.append(P("Executive Data Quality Report: From Dirty Data to Model-Ready Data", ttl))
    S.append(P(f"Tugas Mandiri Pertemuan 2 | Data Mining | {nama} ({nim}) | Dataset: {info['dataset']}", sub))
    S.append(Spacer(1, 3))

    # ------------------------------------------------------------ 1. Data Profile
    S.append(P("1. Data Profile", h1))
    S.append(P(
        f"Dataset <b>{info['dataset']}</b> berisi <b>{info['n_rows']}</b> baris dan <b>{info['n_cols']}</b> kolom "
        f"(klasifikasi biner, target <b>{info['target']}</b>: 1 = selamat, 0 = meninggal). Fitur terdiri dari campuran numerik "
        f"(Age, SibSp, Parch, Fare), kategorikal ordinal (Pclass), kategorikal nominal (Sex, Embarked), serta kolom teks/identitas "
        f"(PassengerId, Name, Ticket, Cabin). Rincian per atribut:"))
    S.append(Spacer(1, 2))
    prof = [["Atribut", "Tipe", "Missing", "Unique", "Peran / keputusan"]] + [list(r) for r in info["profile"]]
    S.append(tbl(prof, [2.4, 1.6, 2.4, 1.4, 10.4]))

    # ------------------------------------------------------------ 2. Data Quality Problems
    S.append(P("2. Data Quality Problems", h1))
    mtxt = ", ".join(f"{k} {v[0]} baris ({v[1]:.2f}%)" for k, v in info["miss_detail"].items())
    S.append(P(
        f"<b>(a) Missing value:</b> {mtxt}. <b>(b) Duplikat:</b> {info['dups']} baris duplikat penuh (data bersih dari duplikasi record). "
        f"<b>(c) Distribusi numerik:</b> Fare sangat right-skewed (skewness {info['fare_skew']:.2f}) dengan outlier tinggi, sedangkan Age "
        f"mendekati simetris (median {info['age_median']:.1f}) sehingga skala antar fitur sangat berbeda. "
        f"<b>(d) Tipe kategorikal:</b> Name/Ticket/Cabin berkardinalitas tinggi dan tidak dapat di-encode langsung; PassengerId hanyalah ID. "
        f"<b>(e) Class imbalance:</b> {info['n_neg']} meninggal vs {info['n_pos']} selamat (rasio {info['ratio']:.2f}:1; kelas minoritas "
        f"{info['pos_pct']:.1f}%), tergolong <i>imbalance ringan</i>."))
    S.append(Spacer(1, 2))
    S.append(Image(fig_path, width=W * cm, height=W * cm * 2.3 / 7.6))

    # ------------------------------------------------------------ 3-4. Treatment & Justification
    S.append(P("3. Treatment dan 4. Justification", h1))
    tr = [["Masalah", "Treatment", "Justifikasi"],
          ["Age missing", "Imputasi <b>median</b> (dihitung dari train saja)",
           "Distribusi Age tidak sepenuhnya simetris dan memiliki outlier; median lebih robust dibanding mean dan tidak menggeser sebaran terlalu jauh."],
          ["Cabin missing sangat tinggi", "Drop Cabin, buat flag biner <b>HasCabin</b>",
           "Mengisi >75% nilai berarti mengarang data; pola ada/tidaknya kabin tetap informatif (terkait kelas tiket)."],
          ["Embarked missing (sangat sedikit)", "Imputasi <b>modus</b> (most_frequent, dari train)",
           "Hanya sedikit baris; modus adalah pilihan paling aman untuk fitur nominal."],
          ["Fitur kategorikal", "Pclass: <b>OrdinalEncoder</b> (1&lt;2&lt;3); Sex, Embarked: <b>OneHotEncoder</b>; drop PassengerId, Name, Ticket",
           "Ordinal mempertahankan urutan kelas; nominal tidak boleh diberi urutan palsu sehingga one-hot; ID/teks unik berisiko overfitting."],
          ["Skala fitur numerik", "<b>StandardScaler</b> pada Age, Fare, SibSp, Parch, FamilySize",
           "KNN berbasis jarak dan Logistic Regression berbasis gradien/regularisasi sensitif skala; tree-based tidak membutuhkannya."],
          ["Feature engineering", "FamilySize = SibSp + Parch + 1",
           "Transformasi per-baris tanpa memakai target atau baris lain, sehingga tidak menimbulkan leakage."],
          ["Class imbalance", "Stratified split; <b>class_weight='balanced'</b> dan <b>SMOTE</b> hanya pada data train",
           "Menjaga proporsi kelas di train/test; SMOTE di luar train membuat sampel sintetis bocor ke test dan menggelembungkan skor."],
          ["Reproducibility & leakage", "ColumnTransformer + Pipeline; split dulu, fit hanya pada train, transform pada test",
           "Statistik imputasi/scaling tidak boleh 'melihat' data test; pipeline memastikan urutan langkah konsisten dan dapat diulang."]]
    S.append(tbl(tr, [3.0, 6.2, 9.0]))

    # ------------------------------------------------------------ 5. Hasil eksperimen
    S.append(P("5. Eksperimen Kecil (observasi dampak preprocessing)", h1))
    if res is None:
        S.append(P("<b>CATATAN DRAFT:</b> angka eksperimen A-D terisi otomatis saat sel terakhir notebook dijalankan "
                   "(Run All). Versi ini hanya memuat fakta data yang tidak bergantung pada hasil model.", note))
        S.append(Spacer(1, 2))
        rows = [["Exp", "Model", "Preprocessing", "Acc", "Prec", "Rec", "F1", "AUC", "CV-F1"],
                ["A", "KNN / LogReg", "Tanpa scaling (baseline)", "-", "-", "-", "-", "-", "-"],
                ["B", "KNN / LogReg", "StandardScaler", "-", "-", "-", "-", "-", "-"],
                ["C", "KNN / LogReg", "Tanpa imbalance handling (baseline)", "-", "-", "-", "-", "-", "-"],
                ["D", "KNN / LogReg", "class_weight / SMOTE", "-", "-", "-", "-", "-", "-"]]
        S.append(tbl(rows, [1.0, 2.6, 6.2, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4]))
    else:
        rows = [["Exp", "Model", "Preprocessing", "Acc", "Prec", "Rec", "F1", "AUC", "CV-F1 (mean+-sd)"]]
        for _, r in res.iterrows():
            rows.append([r.Exp, r.Model, r.Preprocessing, f"{r.Accuracy:.3f}", f"{r.Precision:.3f}", f"{r.Recall:.3f}",
                         f"{r.F1:.3f}", f"{r.ROC_AUC:.3f}", f"{r.CV_F1_mean:.3f}+-{r.CV_F1_std:.3f}"])
        S.append(tbl(rows, [1.0, 1.7, 5.6, 1.3, 1.3, 1.3, 1.3, 1.3, 3.4]))
        S.append(Spacer(1, 2))
        S.append(P(
            f"<b>Interpretasi scaling (A vs B).</b> F1 KNN {delta_word(g('A','KNN','F1'), g('B','KNN','F1'))} "
            f"({g('A','KNN','F1'):.3f} -> {g('B','KNN','F1'):.3f}; Accuracy {g('A','KNN','Accuracy'):.3f} -> {g('B','KNN','Accuracy'):.3f}), "
            f"sedangkan F1 Logistic Regression {delta_word(g('A','LogReg','F1'), g('B','LogReg','F1'))} "
            f"({g('A','LogReg','F1'):.3f} -> {g('B','LogReg','F1'):.3f}). KNN menghitung jarak Euclidean sehingga Fare (0-512) mendominasi "
            f"Age (0-80) dan fitur biner (0/1) bila tidak diskalakan; Logistic Regression lebih toleran tetapi tetap terbantu pada konvergensi dan regularisasi."))
        S.append(Spacer(1, 2))
        S.append(P(
            f"<b>Interpretasi imbalance (C vs D).</b> Pada Logistic Regression, Recall kelas selamat {delta_word(g('C','LogReg','Recall'), g('D1','LogReg','Recall'))} "
            f"dari {g('C','LogReg','Recall'):.3f} menjadi {g('D1','LogReg','Recall'):.3f} (class_weight) dan {g('D2','LogReg','Recall'):.3f} (SMOTE), "
            f"dengan Precision {g('C','LogReg','Precision'):.3f} -> {g('D1','LogReg','Precision'):.3f} / {g('D2','LogReg','Precision'):.3f}. "
            f"Pada KNN, Recall {g('C','KNN','Recall'):.3f} -> {g('D2','KNN','Recall'):.3f} (SMOTE). Persentase prediksi kelas positif berubah dari "
            f"{g('C','LogReg','Pred_Pos_Rate'):.3f} ke {g('D1','LogReg','Pred_Pos_Rate'):.3f} (LogReg + class_weight), yakni model bergeser memprediksi lebih banyak "
            f"kelas minoritas; trade-off-nya adalah Recall vs Precision, bukan sekadar Accuracy. Karena imbalance hanya ringan, "
            f"perubahan metrik yang kecil adalah hasil yang wajar."))
        S.append(Spacer(1, 2))
        S.append(P(
            f"<b>Bukti leakage.</b> Bila SMOTE dijalankan <i>sebelum</i> train-test split, F1 KNN menjadi {info['leak_f1']:.3f} "
            f"dibanding {info['clean_f1']:.3f} pada prosedur yang benar (selisih {info['leak_f1'] - info['clean_f1']:+.3f}); "
            f"skor tersebut menyesatkan karena sampel sintetis di test adalah interpolasi dari sampel train."))

    # ------------------------------------------------------------ 6. Final Dataset Condition
    S.append(P("6. Final Dataset Condition", h1))
    fin = [["Aspek", "Kondisi akhir"],
           ["Dimensi", f"Train {info['train_shape'][0]} x {info['train_shape'][1]}; Test {info['test_shape'][0]} x {info['test_shape'][1]} "
                       f"(fitur: {', '.join(info['final_cols'])})"],
           ["Missing value", "0 pada train dan test (setelah imputasi)"],
           ["Kelas train", f"Asli {info['tr_neg']} : {info['tr_pos']}; setelah SMOTE {info['sm_neg']} : {info['sm_pos']} (test tidak diubah, "
                           f"tetap {info['te_neg']} : {info['te_pos']})"],
           ["Skala", "Fitur numerik terstandardisasi (mean ~0, std ~1 pada train); statistik berasal dari train saja"],
           ["Leakage check", "Split dilakukan paling awal, stratified; imputer/scaler/encoder fit hanya pada train; SMOTE hanya pada train; "
                             "tidak ada indeks yang beririsan antara train dan test"]]
    S.append(tbl(fin, [3.0, 15.2]))
    S.append(Spacer(1, 2))
    S.append(P(
        "<b>Risiko leakage yang diidentifikasi:</b> (1) fit imputer/scaler pada seluruh data sebelum split sehingga statistik test ikut mempengaruhi train; "
        "(2) SMOTE sebelum split; (3) fitur turunan target atau target encoding tanpa cross-validation; (4) anggota keluarga/tiket yang sama terpisah "
        "di train dan test (group leakage khas Titanic). Ketiga yang pertama dicegah oleh urutan pipeline di atas; yang terakhir dicatat sebagai keterbatasan."))

    def footer(canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(colors.grey)
        canvas.drawRightString(A4[0] - 1.4 * cm, 0.9 * cm, f"Halaman {doc.page}")
        canvas.restoreState()

    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=1.4 * cm, rightMargin=1.4 * cm,
                            topMargin=1.2 * cm, bottomMargin=1.4 * cm,
                            title="Executive Data Quality Report", author=nama)
    doc.build(S, onFirstPage=footer, onLaterPages=footer)
    return doc.page

In [ ]:
# ---------- kumpulkan informasi dari notebook ----------
role = {"PassengerId": "Identitas (unik) -> drop", "Survived": "TARGET (0 = meninggal, 1 = selamat)",
        "Pclass": "Ordinal -> OrdinalEncoder", "Name": "Teks unik -> drop", "Sex": "Nominal -> One-Hot",
        "Age": "Numerik -> median impute + scaling", "SibSp": "Numerik -> scaling", "Parch": "Numerik -> scaling",
        "Ticket": "Identitas kardinalitas tinggi -> drop", "Fare": "Numerik skewed -> scaling",
        "Cabin": "Missing tinggi -> drop, buat HasCabin", "Embarked": "Nominal -> modus impute + One-Hot"}
prof_rows = []
for c in df_raw.columns:
    n_miss = int(df_raw[c].isna().sum())
    prof_rows.append((c, str(df_raw[c].dtype).replace("object", "str"),
                      f"{n_miss} ({n_miss/len(df_raw)*100:.1f}%)" if n_miss else "0",
                      int(df_raw[c].nunique()), role.get(c, "-")))

miss_pct = (df_raw.isna().mean() * 100); miss_cnt = df_raw.isna().sum()
miss_pct = miss_pct[miss_pct > 0].sort_values(ascending=False)
info = {
    "dataset": "Titanic", "target": TARGET, "n_rows": len(df_raw), "n_cols": df_raw.shape[1],
    "profile": prof_rows,
    "miss": {k: float(v) for k, v in miss_pct.items()},
    "miss_detail": {k: (int(miss_cnt[k]), float(miss_pct[k])) for k in miss_pct.index},
    "dups": int(dup_full), "fare_skew": float(df_raw["Fare"].skew()), "age_median": float(df_raw["Age"].median()),
    "n_pos": int((y == 1).sum()), "n_neg": int((y == 0).sum()),
    "ratio": float(ratio), "pos_pct": float(y.mean() * 100),
    "tr_pos": int((y_train == 1).sum()), "tr_neg": int((y_train == 0).sum()),
    "te_pos": int((y_test == 1).sum()), "te_neg": int((y_test == 0).sum()),
    "sm_pos": int((np.asarray(y_train_bal) == 1).sum()), "sm_neg": int((np.asarray(y_train_bal) == 0).sum()),
    "train_shape": X_train_prep.shape, "test_shape": X_test_prep.shape, "final_cols": list(X_train_prep.columns),
    "leak_f1": float(leak_f1), "clean_f1": float(clean_f1),
}
REPORT_NAME = f"P02_{NIM}_{NAMA}_Report.pdf"
make_fig(info, res, "output/fig_report.png")
n_pages = build_report(REPORT_NAME, info, res, "output/fig_report.png", nim=NIM, nama=NAMA)
print(f"Laporan dibuat: {REPORT_NAME} | jumlah halaman: {n_pages}")
assert n_pages <= 3, "Laporan melebihi 3 halaman! Kurangi isi atau perkecil font di build_report()."